# Análise Exploratória — Censo Escolar × IDEB (2019, 2021 e 2023)

Este notebook apresenta uma nova análise exploratória das bases utilizadas no projeto, agora com o recorte temporal definido para as primeiras aplicações: **2019, 2021 e 2023**.

O objetivo é:

- caracterizar as duas fontes;
- verificar dimensão, cobertura temporal e distribuição geográfica;
- avaliar valores ausentes e duplicidades;
- identificar códigos especiais;
- analisar a distribuição do IDEB;
- validar a chave de integração;
- cruzar Censo Escolar e IDEB;
- explorar associações entre características escolares e IDEB;
- preparar atributos derivados para análises posteriores.

As primeiras 1.000 linhas são utilizadas apenas para inspeção técnica. As verificações de qualidade e as estatísticas principais são calculadas sobre as bases completas.


## 1. Importação das bibliotecas


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


## 2. Inspeção inicial das primeiras 1.000 linhas

A inspeção inicial permite verificar rapidamente a estrutura dos arquivos antes de carregar as bases completas. Como os arquivos podem estar ordenados, essas 1.000 linhas não devem ser interpretadas como uma amostra estatisticamente representativa.


In [ ]:
censo_amostra = pd.read_csv(
    "../data/censo_nordeste_2019+.csv",
    nrows=1000,
    low_memory=False
)

ideb_amostra = pd.read_csv(
    "../data/ideb_nordeste_2019+.csv",
    nrows=1000,
    low_memory=False
)

print("Censo Escolar - amostra:", censo_amostra.shape)
print("IDEB - amostra:", ideb_amostra.shape)


In [ ]:
print("Distribuição temporal nas primeiras 1.000 linhas do Censo:")
display(censo_amostra["ano"].value_counts().sort_index().to_frame("registros"))

print("\nDistribuição temporal nas primeiras 1.000 linhas do IDEB:")
display(ideb_amostra["ano"].value_counts().sort_index().to_frame("registros"))


In [ ]:
censo_amostra.head()


In [ ]:
ideb_amostra.head()


## 3. Estrutura das bases


In [ ]:
print("Colunas do Censo Escolar:")
print(censo_amostra.columns.tolist())

print("\nColunas do IDEB:")
print(ideb_amostra.columns.tolist())


In [ ]:
print("Tipos de dados - Censo Escolar")
display(censo_amostra.dtypes.to_frame("tipo"))

print("\nTipos de dados - IDEB")
display(ideb_amostra.dtypes.to_frame("tipo"))


## 4. Carregamento completo das bases

A partir desta etapa, as análises são realizadas sobre todos os registros dos dois arquivos.


In [ ]:
censo = pd.read_csv(
    "../data/censo_nordeste_2019+.csv",
    low_memory=False
)

ideb = pd.read_csv(
    "../data/ideb_nordeste_2019+.csv",
    low_memory=False
)

print(f"Censo Escolar: {censo.shape[0]:,} registros e {censo.shape[1]} colunas")
print(f"IDEB: {ideb.shape[0]:,} registros e {ideb.shape[1]} colunas")


## 5. Cobertura temporal


In [ ]:
anos_censo = sorted(censo["ano"].dropna().unique())
anos_ideb = sorted(ideb["ano"].dropna().unique())

print("Anos disponíveis no Censo:", anos_censo)
print("Anos disponíveis no IDEB:", anos_ideb)


In [ ]:
distribuicao_censo_ano = (
    censo["ano"]
    .value_counts()
    .sort_index()
    .rename_axis("ano")
    .reset_index(name="registros")
)

distribuicao_ideb_ano = (
    ideb["ano"]
    .value_counts()
    .sort_index()
    .rename_axis("ano")
    .reset_index(name="registros")
)

print("Censo por ano")
display(distribuicao_censo_ano)

print("IDEB por ano")
display(distribuicao_ideb_ano)


## 6. Distribuição geográfica


In [ ]:
censo_por_estado = (
    censo["sigla_uf"]
    .value_counts()
    .rename_axis("UF")
    .reset_index(name="registros")
)

ideb_por_estado = (
    ideb["sigla_uf"]
    .value_counts()
    .rename_axis("UF")
    .reset_index(name="registros")
)

print("Censo Escolar por estado")
display(censo_por_estado)

print("IDEB por estado")
display(ideb_por_estado)


## 7. Dependência administrativa e localização


In [ ]:
rede = (
    censo["rede"]
    .value_counts(dropna=False)
    .rename_axis("rede")
    .reset_index(name="registros")
)

localizacao = (
    censo["tipo_localizacao"]
    .value_counts(dropna=False)
    .rename_axis("tipo_localizacao")
    .reset_index(name="registros")
)

print("Rede administrativa")
display(rede)

print("Localização")
display(localizacao)


## 8. Valores ausentes no Censo Escolar

São calculados a quantidade e o percentual de valores ausentes por atributo. Como a amostra contempla três anos diferentes, também será analisada a ausência por ano.


In [ ]:
nulos_censo = pd.DataFrame({
    "quantidade_nulos": censo.isna().sum(),
    "percentual_nulos": censo.isna().mean() * 100
}).sort_values("percentual_nulos", ascending=False)

display(nulos_censo)


### 8.1 Valores ausentes do Censo por ano


In [ ]:
colunas_com_nulos_relevantes = (
    nulos_censo[nulos_censo["percentual_nulos"] > 5]
    .index
    .tolist()
)

nulos_censo_por_ano = (
    censo
    .groupby("ano")[colunas_com_nulos_relevantes]
    .apply(lambda grupo: grupo.isna().mean() * 100)
)

display(nulos_censo_por_ano)


## 9. Valores ausentes no IDEB


In [ ]:
nulos_ideb = pd.DataFrame({
    "quantidade_nulos": ideb.isna().sum(),
    "percentual_nulos": ideb.isna().mean() * 100
}).sort_values("percentual_nulos", ascending=False)

display(nulos_ideb)


In [ ]:
nulos_ideb_por_ano = (
    ideb
    .groupby("ano")
    .apply(lambda grupo: grupo.isna().mean() * 100)
)

display(nulos_ideb_por_ano)


## 10. Verificação de duplicidades no Censo Escolar


In [ ]:
duplicados_completos_censo = censo.duplicated().sum()
duplicados_chave_censo = censo.duplicated(
    subset=["ano", "id_escola"]
).sum()

duplicidades_censo = pd.DataFrame({
    "verificacao": [
        "Linhas completamente duplicadas",
        "Duplicidade por ano + id_escola"
    ],
    "quantidade": [
        duplicados_completos_censo,
        duplicados_chave_censo
    ]
})

display(duplicidades_censo)


## 11. Verificação de duplicidades no IDEB

No IDEB, uma escola pode aparecer mais de uma vez no mesmo ano por possuir resultados para diferentes etapas de ensino. Por isso, a combinação `ano + id_escola + anos_escolares` é utilizada para verificar a unicidade real dos registros.


In [ ]:
duplicados_completos_ideb = ideb.duplicated().sum()

duplicados_escola_ano = ideb.duplicated(
    subset=["ano", "id_escola"]
).sum()

duplicados_escola_ano_etapa = ideb.duplicated(
    subset=["ano", "id_escola", "anos_escolares"]
).sum()

duplicidades_ideb = pd.DataFrame({
    "verificacao": [
        "Linhas completamente duplicadas",
        "Repetições por ano + id_escola",
        "Duplicidades por ano + id_escola + anos_escolares"
    ],
    "quantidade": [
        duplicados_completos_ideb,
        duplicados_escola_ano,
        duplicados_escola_ano_etapa
    ]
})

display(duplicidades_ideb)


## 12. Verificação de valores especiais no Censo


In [ ]:
colunas_profissionais = [
    "quantidade_profissional_saude",
    "quantidade_profissional_nutricionista",
    "quantidade_profissional_psicologo",
    "quantidade_profissional_pedagogia"
]

resultado_88888 = []

for coluna in colunas_profissionais:
    if coluna in censo.columns:
        resultado_88888.append({
            "variavel": coluna,
            "quantidade_88888": (censo[coluna] == 88888).sum()
        })

display(pd.DataFrame(resultado_88888))


### 12.1 Tratamento dos códigos especiais

O valor `88888`, quando presente nessas variáveis de quantidade, é convertido para `NaN` antes das estatísticas seguintes, evitando que seja tratado como uma quantidade real.


In [ ]:
for coluna in colunas_profissionais:
    if coluna in censo.columns:
        censo[coluna] = censo[coluna].replace(88888, np.nan)


## 13. Verificação de códigos categóricos


In [ ]:
colunas_categoricas_verificar = [
    "orgao_gremio_estudantil",
    "material_pedagogico_multimidia",
    "material_pedagogico_infantil",
    "material_pedagogico_cientifico",
    "material_pedagogico_musical",
    "material_pedagogico_artistica"
]

for coluna in colunas_categoricas_verificar:
    if coluna in censo.columns:
        print(f"\n{coluna}")
        print(censo[coluna].value_counts(dropna=False).sort_index())


## 14. Caracterização da tabela IDEB


In [ ]:
print("Distribuição por nível de ensino:")
display(
    ideb["ensino"]
    .value_counts()
    .rename_axis("ensino")
    .reset_index(name="registros")
)

print("Distribuição por etapa:")
display(
    ideb["anos_escolares"]
    .value_counts()
    .rename_axis("anos_escolares")
    .reset_index(name="registros")
)


## 15. Estatísticas descritivas do IDEB


In [ ]:
variaveis_ideb = [
    "taxa_aprovacao",
    "indicador_rendimento",
    "nota_saeb_media_padronizada",
    "ideb",
    "projecao"
]

display(ideb[variaveis_ideb].describe().T)


## 16. IDEB por ano


In [ ]:
ideb_por_ano = (
    ideb
    .groupby("ano")["ideb"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reset_index()
)

display(ideb_por_ano)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    ideb_por_ano["ano"],
    ideb_por_ano["mean"],
    marker="o"
)
plt.xlabel("Ano")
plt.ylabel("IDEB médio")
plt.title("IDEB médio por ano")
plt.xticks(ideb_por_ano["ano"])
plt.grid(alpha=0.3)
plt.show()


## 17. IDEB por etapa de ensino


In [ ]:
ideb_por_etapa = (
    ideb
    .groupby("anos_escolares")["ideb"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

display(ideb_por_etapa)


## 18. Distribuição geral do IDEB


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(ideb["ideb"].dropna(), bins=30)
plt.xlabel("IDEB")
plt.ylabel("Quantidade de registros")
plt.title("Distribuição do IDEB")
plt.show()


# Cruzamento entre Censo Escolar e IDEB


## 19. Validação dos anos e da chave de cruzamento

As duas bases já estão restritas aos anos escolhidos para a amostra: 2019, 2021 e 2023. O cruzamento é realizado pela combinação `ano + id_escola`.

No Censo essa combinação deve ser única. No IDEB, uma mesma escola pode aparecer em mais de uma etapa de ensino, caracterizando um relacionamento muitos-para-um do IDEB para o Censo.


In [ ]:
anos_comuns = sorted(
    set(censo["ano"].unique()).intersection(
        set(ideb["ano"].unique())
    )
)

print("Anos comuns:", anos_comuns)

print(
    "Duplicidades ano + id_escola no Censo:",
    censo.duplicated(["ano", "id_escola"]).sum()
)


## 20. Cruzamento das bases


In [ ]:
dados = ideb.merge(
    censo,
    on=["ano", "id_escola"],
    how="left",
    validate="many_to_one",
    indicator=True,
    suffixes=("_ideb", "_censo")
)

print("Dimensão do conjunto integrado:", dados.shape)


## 21. Qualidade do cruzamento


In [ ]:
correspondencia = (
    dados["_merge"]
    .value_counts()
    .rename_axis("resultado")
    .reset_index(name="registros")
)

display(correspondencia)

percentual_correspondencia = (
    (dados["_merge"] == "both").mean() * 100
)

print(f"Correspondência IDEB → Censo: {percentual_correspondencia:.2f}%")


## 22. Correspondência por ano


In [ ]:
cruzamento_por_ano = (
    dados
    .groupby("ano")
    .agg(
        registros_ideb=("id_escola", "size"),
        escolas_distintas=("id_escola", "nunique"),
        encontrados=("_merge", lambda x: (x == "both").sum())
    )
    .reset_index()
)

cruzamento_por_ano["percentual_encontrado"] = (
    cruzamento_por_ano["encontrados"] /
    cruzamento_por_ano["registros_ideb"] * 100
)

display(cruzamento_por_ano)


## 23. Dimensão final do conjunto integrado


In [ ]:
print(f"Observações escola-etapa-ano: {len(dados):,}")

escola_ano = (
    dados[["ano", "id_escola"]]
    .drop_duplicates()
    .shape[0]
)

print(f"Combinações únicas escola-ano: {escola_ano:,}")
print(f"Escolas distintas no conjunto integrado: {dados['id_escola'].nunique():,}")


## 24. Estatísticas do IDEB após o cruzamento


In [ ]:
display(dados["ideb"].describe().to_frame("ideb"))


## 25. IDEB por ano no conjunto integrado


In [ ]:
ideb_cruzado_ano = (
    dados
    .groupby("ano")["ideb"]
    .agg(["count", "mean", "median", "std"])
    .reset_index()
)

display(ideb_cruzado_ano)


## 26. IDEB por etapa no conjunto integrado


In [ ]:
ideb_cruzado_etapa = (
    dados
    .groupby("anos_escolares")["ideb"]
    .agg(["count", "mean", "median", "std"])
    .reset_index()
)

display(ideb_cruzado_etapa)


## 27. IDEB por rede administrativa


In [ ]:
ideb_rede = (
    dados
    .groupby("rede")["ideb"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(ideb_rede)


## 28. IDEB por localização


In [ ]:
ideb_localizacao = (
    dados
    .groupby("tipo_localizacao")["ideb"]
    .agg(["count", "mean", "median", "std"])
    .sort_values("mean", ascending=False)
)

display(ideb_localizacao)


## 29. Características de infraestrutura

O percentual de presença é calculado apenas entre registros válidos. Isso é importante porque algumas variáveis possuem valores ausentes.


In [ ]:
variaveis_infraestrutura = [
    "agua_potavel",
    "agua_rede_publica",
    "energia_rede_publica",
    "esgoto_rede_publica",
    "lixo_servico_coleta",
    "area_verde",
    "banheiro_chuveiro",
    "biblioteca",
    "cozinha",
    "laboratorio_ciencias",
    "laboratorio_informatica",
    "quadra_esportes",
    "refeitorio",
    "sala_leitura",
    "desktop_aluno",
    "internet_alunos"
]

percentuais = []

for coluna in variaveis_infraestrutura:
    if coluna in dados.columns:
        serie = dados[coluna].dropna()

        percentuais.append({
            "variavel": coluna,
            "registros_validos": len(serie),
            "percentual_presenca": (
                serie.mean() * 100 if len(serie) > 0 else np.nan
            )
        })

infraestrutura_df = pd.DataFrame(percentuais)

display(
    infraestrutura_df.sort_values(
        "percentual_presenca",
        ascending=False
    )
)


## 30. IDEB médio segundo características de infraestrutura

Esta comparação é exploratória. Diferenças de média não devem ser interpretadas como relações de causa e efeito, pois outras características da escola podem influenciar simultaneamente o IDEB e a presença desses recursos.


In [ ]:
variaveis_comparacao = [
    "biblioteca",
    "laboratorio_ciencias",
    "laboratorio_informatica",
    "quadra_esportes",
    "internet_alunos",
    "agua_potavel"
]

resultados = []

for coluna in variaveis_comparacao:
    if coluna in dados.columns:
        tabela = (
            dados
            .dropna(subset=[coluna])
            .groupby(coluna)["ideb"]
            .agg(["count", "mean", "median"])
            .reset_index()
        )

        tabela["variavel"] = coluna
        resultados.append(tabela)

if resultados:
    comparacao_infraestrutura = pd.concat(
        resultados,
        ignore_index=True
    )

    display(comparacao_infraestrutura)


## 31. Construção de variáveis relacionadas às matrículas

Valores absolutos são influenciados pelo tamanho da escola. Por isso, são criadas medidas proporcionais para permitir comparações entre escolas de portes diferentes.


In [ ]:
dados["total_matriculas_sexo"] = (
    dados["quantidade_matricula_feminino"].fillna(0) +
    dados["quantidade_matricula_masculino"].fillna(0) +
    dados["quantidade_matricula_nao_declarada"].fillna(0)
)

dados["proporcao_feminino"] = np.where(
    dados["total_matriculas_sexo"] > 0,
    dados["quantidade_matricula_feminino"] /
    dados["total_matriculas_sexo"],
    np.nan
)

display(
    dados[
        [
            "total_matriculas_sexo",
            "proporcao_feminino"
        ]
    ].describe().T
)


## 32. Composição racial das matrículas


In [ ]:
colunas_raca = [
    "quantidade_matricula_branca",
    "quantidade_matricula_preta",
    "quantidade_matricula_parda",
    "quantidade_matricula_amarela",
    "quantidade_matricula_indigena"
]

dados["total_matriculas_raca"] = (
    dados[colunas_raca]
    .fillna(0)
    .sum(axis=1)
)

for coluna in colunas_raca:
    nome = coluna.replace(
        "quantidade_matricula_",
        "proporcao_"
    )

    dados[nome] = np.where(
        dados["total_matriculas_raca"] > 0,
        dados[coluna] / dados["total_matriculas_raca"],
        np.nan
    )

colunas_proporcao_raca = [
    "proporcao_branca",
    "proporcao_preta",
    "proporcao_parda",
    "proporcao_amarela",
    "proporcao_indigena"
]

display(dados[colunas_proporcao_raca].describe().T)


## 33. Correlações exploratórias

As correlações servem como uma primeira medida de associação linear. Elas não representam causalidade e devem ser interpretadas junto com variáveis de controle como ano, rede administrativa, localização e etapa de ensino.


In [ ]:
variaveis_correlacao = [
    "ideb",
    "quantidade_sala_utilizada_climatizada",
    "quantidade_profissional_saude",
    "quantidade_profissional_nutricionista",
    "quantidade_profissional_psicologo",
    "quantidade_profissional_pedagogia",
    "total_matriculas_sexo",
    "proporcao_feminino",
    "proporcao_branca",
    "proporcao_preta",
    "proporcao_parda",
    "proporcao_amarela",
    "proporcao_indigena"
]

variaveis_correlacao = [
    coluna
    for coluna in variaveis_correlacao
    if coluna in dados.columns
]

correlacao = dados[variaveis_correlacao].corr(numeric_only=True)

display(correlacao)


# Síntese da análise exploratória

As duas bases utilizadas nesta versão já estão delimitadas aos anos de **2019, 2021 e 2023**, definidos como amostra inicial do projeto. Esse recorte permite trabalhar com um conjunto mais recente de variáveis do Censo Escolar e mantém compatibilidade temporal direta com as edições do IDEB.

O Censo Escolar utiliza `ano + id_escola` como chave de identificação de uma escola em um determinado ano. Na tabela do IDEB, a mesma escola pode apresentar mais de um registro no ano em razão das diferentes etapas avaliadas, sendo `ano + id_escola + anos_escolares` a combinação adequada para identificar cada observação do indicador.

A avaliação de valores ausentes deve considerar tanto o percentual global quanto sua distribuição por ano. Variáveis com grande quantidade de ausências ou códigos especiais precisam ser tratadas antes de análises estatísticas e modelagem.

O cruzamento é realizado por `ano + id_escola` em uma relação muitos-para-um do IDEB para o Censo. A qualidade dessa integração é avaliada pelo percentual de registros do IDEB que encontram correspondência no Censo.

As análises de infraestrutura, composição das matrículas e correlações são exploratórias. Diferenças observadas entre grupos não devem ser interpretadas isoladamente como relações causais, pois fatores como ano, estado, rede administrativa, localização, etapa de ensino e tamanho da escola podem atuar simultaneamente sobre o desempenho educacional.
